# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata values
md = dataset.metadata
print(f"{md.name}: {md.description}")

# Print summary information
print("\nAuthors (@id):")
pprint.pprint(getattr(md, 'author', []))

print("\nKeywords:", getattr(md, 'keywords', []))

print("\nLicense:", getattr(md, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields, referenced by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

all_record_sets_info = []
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', '[No name]')
    rs_fields = rs.get('field', [])
    if isinstance(rs_fields, dict):  # Single field
        rs_fields = [rs_fields]
    field_ids = [field_ref['@id'] if isinstance(field_ref, dict) and '@id' in field_ref else str(field_ref) for field_ref in rs_fields]
    print(f"Record Set: {rs_name}\n  - @id: {rs_id}\n  - Field @ids: {field_ids}\n")
    all_record_sets_info.append({'record_set_id': rs_id, 'name': rs_name, 'field_ids': field_ids})

# As an extra step, print field details for the first record set
if record_sets:
    fields = dataset.fields(record_set=record_sets[0]["@id"])
    print(f"Fields in first record set {record_sets[0]['@id']}:")
    for f in fields:
        print(f"  Field @id: {f['@id']}, name: {f.get('name', '[No name]')}, dataType: {f.get('dataType', '[No dtype]')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @ids
record_set_ids = [r['record_set_id'] for r in all_record_sets_info]
print("Record sets to extract:", record_set_ids)
dataframes = {}

# Extract each record set into a pandas DataFrame
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        if len(records) == 0:
            print(f"No records for record set {rec_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded record set '{rec_id}' with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for {rec_id}: {e}")

# Show the first record set fields as an example
if len(dataframes) > 0:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demo, select the first record set and a likely numeric field (e.g., PHQ-9 score, GAD-7 total)
if len(dataframes) > 0:
    rec_id = list(dataframes.keys())[0]    # Pick main survey table
    df = dataframes[rec_id]

    # Try to select a numeric field (look for typical psychometric scales)
    likely_numeric_fields = [col for col in df.columns if any(
        substr in col.lower() for substr in ['phq', 'gad', 'score', 'psq', 'age']
    )]
    print(f"Numeric-like fields: {likely_numeric_fields}")
    # Fallback: just pick the first one in the list
    numeric_field = likely_numeric_fields[0] if likely_numeric_fields else df.select_dtypes(include=[np.number]).columns[0]

    print(f"\nAnalyzing numeric field: {numeric_field}\n")

    # Example analysis: filter, normalize and group
    threshold = df[numeric_field].quantile(0.90) if df[numeric_field].dtype != 'object' else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (top 10%): Found {len(filtered_df)} rows.")
    display(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a demographic column (e.g., gender, education)
    possible_groups = [col for col in df.columns if any(k in col.lower() for k in ['gender', 'education', 'marital', 'village'])]
    group_field = possible_groups[0] if possible_groups else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"\nAverage {numeric_field} by {group_field} for filtered records:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the chosen numeric field
if len(dataframes) > 0:
    df = dataframes[rec_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If group_field found, boxplot by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used the `mlcroissant` library to load, overview, and analyze the FAIR² Kilifi County Mental Health Survey dataset via its Croissant schema. After identifying available record sets and fields (referenced by their `@id` per Croissant best practice), we extracted the main survey DataFrame and performed basic exploratory analysis on numeric mental health assessment fields (e.g., PHQ scores, GAD-7 totals).

Key steps included:
- Programmatic record set and field discovery using `@id`
- Filtering survey responses by high assessment scores, normalization, and grouping by demographics
- Visualizing score distributions and their relationship to group attributes like gender or education level

This workflow can be readily extended for feature engineering, advanced ML analysis, or integration with other FAIR datasets. For more context on the fields and detailed data dictionary, consult the Croissant metadata available at the [source URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).